# Ensemble Learning: 

## 1. Setup and data loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, balanced_accuracy_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams["figure.dpi"] = 110


In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target  # 0 = malignant, 1 = benign  
MALIGNANT = 0
BENIGN = 1
print("Samples:", X.shape[0], " Features:", X.shape[1])
print("Class balance (0=malignant, 1=benign):")
print(pd.Series(y).value_counts().rename(index={0: "malignant", 1: "benign"}))


## 2. Train/test split

We use a stratified split so the malignant/benign ratio is preserved in both
sets, and we keep a fixed random state throughout for reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

print("Train size:", X_train.shape[0], " Test size:", X_test.shape[0])
print("Malignant cases in test set:", (y_test == MALIGNANT).sum(), "/", len(y_test))


## 3. Models

We compare five classifiers:

- A single **Decision Tree** (the baseline non-ensemble model).
- **Random Forest** (bagging).
- **AdaBoost** (boosting).
- **Gradient Boosting** (boosting).
- A **Stacking** ensemble (SVC + Random Forest as base learners, Logistic
  Regression as the meta-learner).

All models use `random_state=RANDOM_STATE` where applicable, and the
tree-based models are left at reasonable default depths rather than
aggressively tuned, since the point here is to compare families of methods,
not to squeeze out the last percentage point.

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "Stacking": StackingClassifier(
        estimators=[
            ("svc", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)),
            ("rf", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
        ],
        final_estimator=LogisticRegression(max_iter=2000),
        cv=5,
    ),
}

for name, model in models.items():
    model.fit(X_train, y_train)
print("All models trained.")


## 4. Evaluation at the default threshold 

We use **ROC-AUC together with balanced accuracy** to select the
best model.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []

# Trivial baseline row
rows.append({
    "Model": "Trivial (always malignant)",
    "Accuracy": trivial_accuracy,
    "Sensitivity": trivial_sensitivity,
    "Specificity": trivial_specificity,
    "F1-score": f1_score(y_test, trivial_pred, pos_label=MALIGNANT),
    "Balanced accuracy": balanced_accuracy_score(y_test, trivial_pred),
    "ROC-AUC": np.nan,
    "CV Sensitivity (mean)": np.nan,
    "Distinct predicted probabilities": np.nan,
})

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, MALIGNANT]

    acc = accuracy_score(y_test, y_pred)
    sens = recall_score(y_test, y_pred, pos_label=MALIGNANT)
    spec = recall_score(y_test, y_pred, pos_label=BENIGN)
    f1 = f1_score(y_test, y_pred, pos_label=MALIGNANT)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    auc = roc_auc_score((y_test == MALIGNANT).astype(int), y_proba)

    rows.append({
        "Model": name,
        "Accuracy": acc,
        "Sensitivity": sens,
        "Specificity": spec,
        "F1-score": f1,
        "Balanced accuracy": bal_acc,
        "ROC-AUC": auc,
        "CV Sensitivity (mean)": np.nan,  # filled in the next cell with the correct pos_label
        "Distinct predicted probabilities": len(np.unique(np.round(y_proba, 6))),
    })

results_df = pd.DataFrame(rows)
results_df


The cross-validation call above needs `pos_label=MALIGNANT` for
sensitivity, which `cross_val_score`'s built-in `"recall"` scorer does not
support directly (it assumes the positive class is label `1`, which here is
benign). We fix this with an explicit scorer.

In [ ]:
from sklearn.metrics import make_scorer

sensitivity_scorer = make_scorer(recall_score, pos_label=MALIGNANT)

cv_means = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=sensitivity_scorer)
    cv_means[name] = scores.mean()

results_df.loc[results_df["Model"] != "Trivial (always malignant)", "CV Sensitivity (mean)"] = \
    results_df.loc[results_df["Model"] != "Trivial (always malignant)", "Model"].map(cv_means)

results_df_sorted = pd.concat([
    results_df[results_df["Model"] == "Trivial (always malignant)"],
    results_df[results_df["Model"] != "Trivial (always malignant)"].sort_values("ROC-AUC", ascending=False),
])
results_df_sorted.reset_index(drop=True)


### Ranking models: ROC-AUC and balanced accuracy together


In [ ]:
ranking_df = results_df[results_df["Model"] != "Trivial (always malignant)"].copy()
ranking_df = ranking_df.sort_values(["ROC-AUC", "Balanced accuracy"], ascending=False)
ranking_df[["Model", "ROC-AUC", "Balanced accuracy", "F1-score", "Sensitivity", "Specificity"]].reset_index(drop=True)


## 5. ROC curves

Plotting the full ROC curve for every model shows the sensitivity/specificity
trade-off across all thresholds, not just the default one.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, MALIGNANT]
    fpr, tpr, _ = roc_curve((y_test == MALIGNANT).astype(int), y_proba)
    auc = roc_auc_score((y_test == MALIGNANT).astype(int), y_proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc:.4f})")

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance level")
ax.set_xlabel("False positive rate (1 - specificity)")
ax.set_ylabel("True positive rate (sensitivity)")
ax.set_title("ROC curves: single tree vs. ensembles")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150)
plt.show()


## 6. Threshold tuning on the best-AUC model

In [ ]:
best_model_name = ranking_df.iloc[0]["Model"]  # top of the ROC-AUC + balanced-accuracy ranking
best_model = models[best_model_name]
print("Selected model (best ROC-AUC, balanced accuracy as tie-breaker):", best_model_name)

y_proba_best = best_model.predict_proba(X_test)[:, MALIGNANT]

thresholds_to_try = [0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1, 0.05, 0.02, 0.01, 0.005, 0.001]
threshold_rows = []

for t in thresholds_to_try:
    y_pred_t = (y_proba_best >= t).astype(int)
    y_pred_t = np.where(y_pred_t == 1, MALIGNANT, BENIGN)  # >= t means "predict malignant"
    sens_t = recall_score(y_test, y_pred_t, pos_label=MALIGNANT)
    spec_t = recall_score(y_test, y_pred_t, pos_label=BENIGN)
    cm = confusion_matrix(y_test, y_pred_t, labels=[MALIGNANT, BENIGN])
    fn = cm[0, 1]  # malignant predicted as benign: the dangerous error
    fp = cm[1, 0]  # benign predicted as malignant: the costly-but-safe error
    threshold_rows.append({
        "Threshold": t, "Sensitivity": sens_t, "Specificity": spec_t,
        "False negatives": fn, "False positives": fp,
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df


In [ ]:
reaches_full_sensitivity = threshold_df.loc[threshold_df["Sensitivity"] >= 0.999, "Threshold"]

if len(reaches_full_sensitivity) > 0:
    chosen_threshold = reaches_full_sensitivity.max()
    note = ""
else:
    chosen_threshold = threshold_df["Threshold"].min()
    note = " (none of the tested thresholds reached 99.9% sensitivity; using the lowest one tried)"

chosen_row = threshold_df.loc[threshold_df["Threshold"] == chosen_threshold].iloc[0]
print(f"Chosen threshold: {chosen_threshold}{note}")
print(chosen_row)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(threshold_df["Threshold"], threshold_df["Sensitivity"], marker="o", label="Sensitivity")
ax.plot(threshold_df["Threshold"], threshold_df["Specificity"], marker="o", label="Specificity")
ax.axvline(chosen_threshold, color="gray", linestyle="--", label=f"Chosen threshold = {chosen_threshold}")
ax.set_xscale("log")
ax.set_xlabel("Decision threshold (log scale)")
ax.set_ylabel("Score")
ax.set_title(f"Sensitivity/specificity trade-off ({best_model_name})")
ax.legend()
plt.tight_layout()
plt.savefig("threshold_tradeoff.png", dpi=150)
plt.show()


### Confusion matrix at the tuned threshold

In [ ]:
y_pred_tuned = (y_proba_best >= chosen_threshold).astype(int)
y_pred_tuned = np.where(y_pred_tuned == 1, MALIGNANT, BENIGN)

cm_final = confusion_matrix(y_test, y_pred_tuned, labels=[MALIGNANT, BENIGN])

fig, ax = plt.subplots(figsize=(6, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_final, display_labels=["malignant", "benign"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"{best_model_name}, threshold {chosen_threshold}\n(tuned for sensitivity)", fontsize=11)
plt.tight_layout()
plt.savefig("confusion_matrix_tuned.png", dpi=150)
plt.show()


## 7. Comparing the tuned ensembles against each other

Since more than one ensemble reaches a near-identical top ROC-AUC, it is
worth checking whether the model with the single highest AUC is actually the
best deployment choice, or whether a very close runner-up reaches the same
safety target with a gentler (less aggressive) threshold.

In [ ]:
top2 = ranking_df.head(2)
print(top2[["Model", "ROC-AUC", "Balanced accuracy"]].to_string(index=False))

runner_up_name = top2.iloc[1]["Model"]
runner_up_model = models[runner_up_name]
y_proba_runner_up = runner_up_model.predict_proba(X_test)[:, MALIGNANT]

runner_up_rows = []
for t in thresholds_to_try:
    y_pred_t = (y_proba_runner_up >= t).astype(int)
    y_pred_t = np.where(y_pred_t == 1, MALIGNANT, BENIGN)
    sens_t = recall_score(y_test, y_pred_t, pos_label=MALIGNANT)
    spec_t = recall_score(y_test, y_pred_t, pos_label=BENIGN)
    cm = confusion_matrix(y_test, y_pred_t, labels=[MALIGNANT, BENIGN])
    runner_up_rows.append({
        "Threshold": t, "Sensitivity": sens_t, "Specificity": spec_t,
        "False negatives": cm[0, 1], "False positives": cm[1, 0],
    })

runner_up_df = pd.DataFrame(runner_up_rows)
runner_up_full_sens = runner_up_df.loc[runner_up_df["Sensitivity"] >= 0.999, "Threshold"]
runner_up_threshold = runner_up_full_sens.max() if len(runner_up_full_sens) > 0 else runner_up_df["Threshold"].min()

print(f"\n{best_model_name}: needs threshold {chosen_threshold} for full sensitivity "
      f"({chosen_row['False positives']} false positives)")
print(f"{runner_up_name}: needs threshold {runner_up_threshold} for full sensitivity")
